# ✈️ HighCloud Airlines – Flight Delay Analysis

## Project Overview
This project analyzes airline flight delays, cancellations, and operational performance using Python and data visualization.

## Objectives
- Calculate key performance indicators (KPIs).
- Compare weekday and weekend flight statistics.
- Analyze monthly arrival delays.
- Compare cancellation rates by airline.
- Identify major causes of flight delays.
- Compare weekly departure and arrival delays.

## Tools & Technologies
Python | Pandas | NumPy | Matplotlib | Kaggle Notebook

## Project Outcome
Interactive-style visualizations and data-driven summaries to understand flight delays and airline performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob

print("Libraries imported successfully!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

print("CSV files found:")

for file in files:
    print(file)

In [ ]:
flight_files = [
    file for file in files
    if os.path.basename(file) == "flights_cleaned.csv"
]

if flight_files:
    df = pd.read_csv(flight_files[0], low_memory=False)

    print("Dataset loaded successfully!")
    print("Total Rows:", len(df))
    print("Total Columns:", len(df.columns))

    display(df.head())
else:
    print("flights_cleaned.csv file not found!")

In [ ]:
import matplotlib.pyplot as plt

total_flights = len(df)

cancelled_flights = df["CANCELLED"].sum()

cancellation_rate = (
    cancelled_flights / total_flights * 100
    if total_flights > 0 else 0
)

avg_arrival_delay = df.loc[
    df["CANCELLED"] == 0, "ARRIVAL_DELAY"
].mean()

kpis = [
    ("TOTAL FLIGHTS", f"{total_flights:,.0f}"),
    ("CANCELLED FLIGHTS", f"{cancelled_flights:,.0f}"),
    ("CANCELLATION RATE", f"{cancellation_rate:.2f}%"),
    ("AVG ARRIVAL DELAY", f"{avg_arrival_delay:.2f} min")
]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.patch.set_facecolor("#F3F7FC")

for ax, (title, value) in zip(axes, kpis):
    ax.set_facecolor("#FFFFFF")

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#B7CDE8")
        spine.set_linewidth(1.5)

    ax.set_xticks([])
    ax.set_yticks([])

    ax.text(
        0.5, 0.68, title,
        ha="center", va="center",
        fontsize=15, fontweight="bold",
        color="#345477", wrap=True
    )

    ax.text(
        0.5, 0.38, value,
        ha="center", va="center",
        fontsize=19, fontweight="bold",
        color="#1769AA"
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle(
    "HIGHCLOUD AIRLINES — KEY PERFORMANCE INDICATORS",
    fontsize=15, fontweight="bold",
    color="#17365D", y=1.05
)

plt.tight_layout()
plt.show()

In [ ]:
day_counts = df["DAY_OF_WEEK"].map(
    lambda x: "Weekend" if x in [6, 7] else "Weekday"
).value_counts()

plt.figure(figsize=(7, 7))
plt.pie(
    day_counts.values,
    labels=day_counts.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Weekday vs Weekend Total Flights")
plt.show()

display(day_counts.rename_axis("Day Type").reset_index(name="Total Flights"))

In [ ]:
df["Flight_Date"] = pd.to_datetime(
    df[["YEAR", "MONTH", "DAY"]].rename(
        columns={"YEAR": "year", "MONTH": "month", "DAY": "day"}
    ),
    errors="coerce"
)

monthly_delay = (
    df[df["CANCELLED"] == 0]
    .groupby(df["Flight_Date"].dt.month)["ARRIVAL_DELAY"]
    .mean()
    .dropna()
)

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.figure(figsize=(12, 5))
plt.plot(
    [month_names[int(m)-1] for m in monthly_delay.index],
    monthly_delay.values,
    marker="o"
)
plt.title("Monthly Average Arrival Delay")
plt.xlabel("Month")
plt.ylabel("Average Arrival Delay (Minutes)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
airline_stats = (
    df.groupby("AIRLINE")
    .agg(
        Total_Flights=("CANCELLED", "size"),
        Cancelled_Flights=("CANCELLED", "sum")
    )
)

airline_stats["Cancellation_Rate"] = (
    airline_stats["Cancelled_Flights"]
    / airline_stats["Total_Flights"] * 100
)

airline_stats = airline_stats.sort_values(
    "Cancellation_Rate", ascending=False
)

plt.figure(figsize=(10, 6))
plt.barh(
    airline_stats.index.astype(str),
    airline_stats["Cancellation_Rate"]
)
plt.gca().invert_yaxis()
plt.title("Cancellation Rate by Airline")
plt.xlabel("Cancellation Rate (%)")
plt.ylabel("Airline Code")
plt.tight_layout()
plt.show()

display(airline_stats.round(2))

In [ ]:
delay_columns = {
    "AIRLINE_DELAY": "Airline Delay",
    "WEATHER_DELAY": "Weather Delay",
    "AIR_SYSTEM_DELAY": "Air System Delay",
    "SECURITY_DELAY": "Security Delay",
    "LATE_AIRCRAFT_DELAY": "Late Aircraft Delay"
}

available = {
    col: label for col, label in delay_columns.items()
    if col in df.columns
}

if available:
    cause_totals = df[list(available)].apply(
        pd.to_numeric, errors="coerce"
    ).clip(lower=0).sum().rename(index=available)

    cause_totals = cause_totals.sort_values()

    plt.figure(figsize=(10, 6))
    plt.barh(cause_totals.index, cause_totals.values)
    plt.title("Total Delay Minutes by Cause")
    plt.xlabel("Total Delay Minutes")
    plt.ylabel("Delay Cause")
    plt.tight_layout()
    plt.show()

    display(cause_totals.to_frame("Total Delay Minutes"))
else:
    print("Delay cause columns are not available in this dataset.")

In [ ]:
weekly_df = df[df["CANCELLED"] == 0].copy()

weekly_df["Week_Start"] = (
    weekly_df["Flight_Date"]
    - pd.to_timedelta(weekly_df["Flight_Date"].dt.dayofweek, unit="D")
)

weekly_delay = (
    weekly_df.groupby("Week_Start")
    [["DEPARTURE_DELAY", "ARRIVAL_DELAY"]]
    .mean()
    .dropna()
)

weekly_delay.columns = [
    "Avg Departure Delay",
    "Avg Arrival Delay"
]

# Display selected weeks for a readable chart
weekly_plot = weekly_delay.head(20)

weekly_plot.plot(
    kind="bar",
    figsize=(14, 6),
    width=0.8
)

plt.title("Weekly Departure vs Arrival Delay")
plt.xlabel("Week Starting Date")
plt.ylabel("Average Delay (Minutes)")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

display(weekly_delay.head(10).round(2))